<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025s1/blob/main/week14_RL/shift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The Shift from scaling training-time compute to scaling testing-time compute

### GPT-2
GPT-2 is trained on a huge dataset of internet text to predict the next word in a sequence (language modeling). 
- Input text tokens are embedded and passed through multiple transformer decoder layers.
- The output is a probability distribution over the vocabulary for the next token.
- Generation is done by sampling tokens one at a time from this distribution.

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

tokenizer = AutoTokenizer.from_pretrained('gpt2-large')
model = GPT2LMHeadModel.from_pretrained('gpt2-large')



In [ ]:
math_question = "What is the coefficient of $x^2y^6$ in the expansion of $\left(\frac{3}{5}x-\frac{y}{2}\right)^8$? Express your answer as a common fraction."
input_ids = tokenizer(math_question, return_tensors='pt').input_ids
output = model.generate(input_ids, do_sample=True, max_length=512, top_p=0.95, top_k=0)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(output[0], skip_special_tokens=True))
print("" + 100 * '-')

### DeepSeek R1

can help write the code, demostrate the shift from scaling training-time compute to scaling test-time compute using DeepSeek-R1, hugging face API call to solve the same math question 

In [12]:
# %%capture
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install flash_attn

import torch
from transformers import (
    AutoTokenizer,  
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)


     ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
     ---------------------------------------- 2.7/2.7 MB 38.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      fatal: not a git repository (or any of the parent directories): .git
      
      
      torch.__version__  = 2.4.1+cpu
      
      
      C:\Users\liangnanyi\AppData\Local\Temp\pip-install-2n4s44rr\flash-attn_0f839409761849b886f9cd5866909513\setup.py:99: UserWarning: flash_attn was requested, but nvcc was not found.  Are you sure your environment has nvcc available?  If you're installing within a container from https://hub.docker.com/r/pytorch/pytorch, only images whose names contain 'devel' will provide nvcc.
        warnings.warn(
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 34, in <module>
        File "C:\Users\liangnanyi\AppData\Local\Temp\pip-install-2n4s44rr\flash-attn_0f839409761849b886f9cd5866909513\setup.py", line 183, in <module>
          CUDAE

In [11]:
model_id = "deepseek-ai/DeepSeek-R1-0528"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,             # FP4 weights
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_cfg,
    device_map="auto",             # put layers on all available GPUs
    trust_remote_code=True         # DeepSeek ships a custom modeling file
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    device_map="auto",
    torch_dtype=torch.float16,
)


c:\Users\liangnanyi\.conda\envs\it3103env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\liangnanyi\.cache\huggingface\hub\models--deepseek-ai--DeepSeek-R1-0528. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Special tokens have been added in the vocabulary, make sure the associated word embeddings ar

ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`

In [ ]:
math_question = (
    "What is the coefficient of x^2 y^6 in the expansion "
    "of ((3/5)x - (y/2))^8? Express your answer as a common fraction."
)

resp = generator(
    math_question,
    max_new_tokens=64,
    top_p=0.95,
    temperature=0.2,
)[0]["generated_text"]

print(resp)
